# 3. Distances between recordings, and their fusion

Each recording now has, at each of the 4 scales, 5 fields with 6 extended-persistence diagrams each. This notebook shows
how two recordings are compared, how the 20 resulting distance matrices are combined, and why the combination introduces
no free parameter.

In [ ]:
import sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "src"), str(ROOT)]
import numpy as np
import matplotlib.pyplot as plt
import figlib as L
L.style()
PAPER = ROOT / "results" / "epuck" / "paper"
RES = ROOT / "results" / "epuck"
load = lambda p: pickle.load(open(p, "rb"))

## Sliced-Wasserstein distance between two diagrams

The distance of Carrière, Cuturi and Oudot (2017), of order one. Each diagram is augmented with the projections onto the
diagonal of the points of the other one, so both have the same number of points; both are projected on 25 directions, the
projections are sorted, and the absolute differences are averaged over the directions. The six diagrams of a field are
combined by their sum:

$$d_{f,m}(A,B)=\sum_{s=1}^{6}\mathrm{SW}\big(D^s_{f,m}(A),D^s_{f,m}(B)\big).$$

The cell below recomputes one entry with a 15-line implementation and compares it with the stored matrix.

In [ ]:
import run_sweep as RS
lab, rk, ctrl = RS.load_labels()
D, ladder = RS.load_distances()                       # 20 matrices, 2500 x 2500
def sw(A, B, n_dirs=25):
    A = np.asarray(A, float).reshape(-1, 2); B = np.asarray(B, float).reshape(-1, 2)
    th = np.linspace(-np.pi / 2, np.pi / 2, n_dirs, endpoint=False); u = np.stack([np.cos(th), np.sin(th)], 1)
    dA = 0.5 * (A[:, 0] + A[:, 1]); dB = 0.5 * (B[:, 0] + B[:, 1])
    PA = np.concatenate([A @ u.T, (dB[:, None] * (u[:, 0] + u[:, 1])[None, :])], 0)
    PB = np.concatenate([B @ u.T, (dA[:, None] * (u[:, 0] + u[:, 1])[None, :])], 0)
    return float(np.abs(np.sort(PA, 0) - np.sort(PB, 0)).sum(0).mean())
i, j, f, si = 0, 1000, "rho", 2
gi = lambda ti: load(RES / "stage2_alpha_dgms" / f"traj{ti:04d}_s{si}.pkl")["dgms"][f]
mine = sum(sw(a, b) for a, b in zip(gi(i), gi(j)))
print(f"recomputed d = {mine:.6f}   stored d = {D[f][si][i, j]:.6f}")

## The 20 matrices on Top-1

Rows: fields; columns: scales. Each matrix is shown as a similarity after rank normalisation (dark = similar). The runs are
ordered by mission **for display only**; the method never sees this order. Block structure is already visible in single
views, but every view also mixes some pairs.

In [ ]:
from classification import fusion as F
idx = RS.subset(1, rk)
Q = F.rank_normalise(D, F.ALL_FIELDS, idx)
fig, axes = plt.subplots(5, 4, figsize=(8, 9.5))
for a, f in enumerate(F.ALL_FIELDS):
    for b, s in enumerate(Q[f]):
        axes[a, b].imshow(1 - Q[f][s], cmap=L.SEQ); axes[a, b].set_xticks([]); axes[a, b].set_yticks([])
        if b == 0: axes[a, b].set_ylabel(f)
        if a == 0: axes[a, b].set_title(f"sigma_{b+1}")
plt.tight_layout(); plt.show()

## Rank normalisation

The raw distances of different fields live on different scales and distributions. Each pair of runs first gets its
**mutual rank** (Obayashi and Kinoshita, 2009): the geometric mean of the rank of B among the neighbours of A and of
the rank of A among the neighbours of B. A pair is close only when each run is among the nearest of the other, whatever
the density of its neighbourhood, so a run near a tight group is not pulled into it. The mutual ranks are then replaced
by their percentile ranks, which gives every matrix the same (uniform) distribution, so no field can dominate by the
size of its numbers.

In [ ]:
iu = np.triu_indices(len(idx), 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for f in F.ALL_FIELDS:
    raw = np.asarray(D[f][2], float)[np.ix_(idx, idx)][iu]
    axes[0].hist(raw / raw.max(), bins=60, histtype="step", label=f); axes[1].hist(Q[f][2][iu], bins=60, histtype="step", label=f)
axes[0].set_title("raw distances (scaled by their max)"); axes[1].set_title("after rank normalisation"); axes[0].legend(); plt.show()

## Balanced views: one per scale

The fields come from three moments: density ($M_0$), kinetic intensity ($M_2$), flux ($\varphi,\psi,J$ from $M_1$).
Equal weight per moment, shared equally inside the flux, is the only rule that favours no moment and adds no
parameter:

$$W_m = 1-\Big(\tfrac13\tilde d_{\rho,m}+\tfrac13\tilde d_{K,m}+\tfrac19(\tilde d_{\varphi,m}+\tilde d_{\psi,m}+\tilde d_{J,m})\Big).$$

In [ ]:
print("weights:", F.source_weights(F.ALL_FIELDS))
views = F.scale_views(Q)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for k, v in enumerate(views):
    axes[k].imshow(v, cmap=L.SEQ); axes[k].set_title(f"W_{k+1}"); axes[k].set_xticks([]); axes[k].set_yticks([])
plt.show()

## Diffusion across scales (similarity network fusion)

Each view keeps the $k_{nn}$ nearest neighbours of every run and is diffused $t$ times through the average of the other
views. A link survives only if two runs are neighbours **at several scales**. The pair $(k_{nn},t)$ is the resolution of
the graph: small values keep fine structure, large values merge neighbouring groups. The kernel keeps distinct
behaviours apart, and the diffusion joins the runs of one behaviour that single scales leave split.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (k, t) in zip(axes, [(3, 1), (7, 10), (25, 25)]):
    S = F.consensus(views, k, t)
    ax.imshow(S, cmap=L.SEQ, vmax=np.quantile(S, 0.995)); ax.set_title(f"S*  k_nn={k}, t={t}")
    ax.set_xticks([]); ax.set_yticks([])
plt.show()